# Thesis Notebook

## Install dependencies

In [1]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
!pip install --upgrade transformers datasets peft bitsandbytes accelerate evaluate seqeval fsspec huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 121.7 MB/s eta 0:00:00
  DEPRECATION: Building 'seqeval' using the legacy setup.py bdist_wheel mecha

In [ ]:
# check gpu driver
!nvidia-smi

Thu Jul 10 11:07:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Training

### Training BERT with LoRA for Question Answering

#### Main Experiment Function

In [5]:
# === qa_lora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc
import json # Import json

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
)


def run_lora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 10,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
    skip_training: bool = False, # Added parameter to skip training
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    - skip_training: If True, skips training and proceeds directly to evaluation.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)

    # Check if preprocessed files exist before mapping
    train_processed_path = os.path.join(output_dir, f"lora_train_ds_seed_{seed}.pt")
    val_processed_path = os.path.join(output_dir, f"lora_val_ds_seed_{seed}.pt")

    if os.path.exists(train_processed_path) and os.path.exists(val_processed_path):
        print("Loading preprocessed datasets...")
        train_ds = torch.load(train_processed_path)
        val_ds = torch.load(val_processed_path)
    else:
        print("Preprocessing datasets...")
        train_ds = raw["train"].map(preprocess_fn, batched=False)
        val_ds = raw["validation"].map(preprocess_fn, batched=False)
        train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        # Save preprocessed datasets
        os.makedirs(output_dir, exist_ok=True)
        torch.save(train_ds, train_processed_path)
        torch.save(val_ds, val_processed_path)


    # Model + LoRA setup
    model = BertForQuestionAnswering.from_pretrained(base_model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="QUESTION_ANS",
        use_rslora=False,
    )

    model_save_path = os.path.join(output_dir, f"lora_squad_seed_{seed}")

    if not skip_training:
        model = get_peft_model(model, lora_cfg)

        # Training arguments
        train_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"seed_{seed}"),
            per_device_train_batch_size=batch_size,
            num_train_epochs=num_epochs,
            learning_rate=learning_rate,
            fp16=torch.cuda.is_available(),
            logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
            logging_strategy="steps",
            logging_steps=500,
            save_strategy="epoch",
            eval_strategy="epoch",
            report_to="none",
            label_names=["start_positions", "end_positions"],
        )
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=default_data_collator,
        )

        # Training
        print(f"Training with seed {seed}...")
        trainer.train()

        # Check peak memory
        peak_mem = None
        if torch.cuda.is_available():
            peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
            print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

        # Save model and tokenizer
        trainer.save_model(model_save_path)
        tokenizer.save_pretrained(model_save_path)

        # Cleanup training objects
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()


    # Evaluate
    print("Evaluating model...")
    # Load LoRA-wrapped model for inference
    eval_base_model = BertForQuestionAnswering.from_pretrained(base_model)

    # Check if the saved model exists before loading
    if not os.path.exists(model_save_path):
        raise FileNotFoundError(f"Model not found at {model_save_path}. Please train the model first.")

    infer_model = PeftModel.from_pretrained(eval_base_model, model_save_path)

    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    evaluation_details = [] # List to store evaluation details

    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predicted_answer = out.get("answer", "")
        predictions.append({"id": item["id"], "prediction_text": predicted_answer})
        references.append({"id": item["id"], "answers": item["answers"]})

        # Store evaluation details
        evaluation_details.append({
            "id": item["id"],
            "context": item["context"],
            "question": item["question"],
            "predicted_answer": predicted_answer,
            "gold_answer": item["answers"]["text"][0] if item["answers"]["text"] else "" # Assuming there's at least one gold answer
        })


    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del infer_model, eval_base_model
    gc.collect()

    # Save results to a JSON file
    results_file_path = os.path.join(output_dir, f"lora_results_seed_{seed}.json")
    with open(results_file_path, "w") as f:
        json.dump(results, f, indent=4)

    # Save evaluation details to a JSON file
    evaluation_details_path = os.path.join(output_dir, f"lora_evaluation_details_seed_{seed}.json")
    with open(evaluation_details_path, "w") as f:
        json.dump(evaluation_details, f, indent=4)


    return results

## Seed 42 (10 epochs)


In [6]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed = 42)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 42...


Epoch,Training Loss,Validation Loss
1,2.648100,2.383413
2,2.326400,2.107742
3,2.105000,1.903404
4,1.958700,1.802890
5,1.887600,1.730323
6,1.812000,1.681460
7,1.766800,1.652022
8,1.749500,1.634471
9,1.740500,1.614088
10,1.727300,1.613744


Peak CUDA memory (GB): 4.34
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 67.16 / 76.68
{'exact_match': 67.16177861873226, 'f1': 76.67940694268002}


#### Seed 42 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed = 42)
print(results)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

...

Training with seed 42...


Epoch,Training Loss,Validation Loss
1,2.698000,2.437618
2,2.428000,2.232366
3,2.345400,2.173697


Peak CUDA memory (GB): 4.36
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 50.19 / 60.04
{'exact_match': 50.189214758751184, 'f1': 60.03738370724641}


#### Seed 1234 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=1234)
print(results)

Preprocessing datasets...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 1234...


Epoch,Training Loss,Validation Loss
1,2.676900,2.323775
2,2.302500,2.075966
3,2.207200,2.012451


Peak CUDA memory (GB): 4.32
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 57.49 / 68.18
{'exact_match': 57.49290444654683, 'f1': 68.17750289786743}


#### Seed 2023 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2023)
print(results)

Preprocessing datasets...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 2023...


Epoch,Training Loss,Validation Loss
1,2.681200,2.369022
2,2.343000,2.124326
3,2.240800,2.053578


Peak CUDA memory (GB): 4.32
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 56.26 / 66.37
{'exact_match': 56.26300851466414, 'f1': 66.36561854962821}


#### Seed 2024 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2024)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 2024...


Epoch,Training Loss,Validation Loss
1,2.663600,2.375451
2,2.333200,2.116294
3,2.233800,2.050821


Peak CUDA memory (GB): 4.44
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 56.51 / 66.85
{'exact_match': 56.50898770104068, 'f1': 66.84889504461492}


#### Seed 2025 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_lora_experiment(seed=2025)
print(results)

Preprocessing datasets...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 2025...


Epoch,Training Loss,Validation Loss
1,2.637800,2.314611
2,2.276800,2.057589
3,2.184000,1.997960


Peak CUDA memory (GB): 4.73
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 57.82 / 68.35
{'exact_match': 57.8240302743614, 'f1': 68.35156912315153}


### Training BERT with QLoRA for Question Answering

#### Main Experiment Function

In [ ]:
# === qa_qlora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc
import json # Import json

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training
)


def run_qlora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
    skip_training: bool = False, # Added parameter to skip training
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    - skip_training: If True, skips training and proceeds directly to evaluation.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)

    # Check if preprocessed files exist before mapping
    train_processed_path = os.path.join(output_dir, f"qlora_train_ds_seed_{seed}.pt")
    val_processed_path = os.path.join(output_dir, f"qlora_val_ds_seed_{seed}.pt")

    if os.path.exists(train_processed_path) and os.path.exists(val_processed_path):
        print("Loading preprocessed datasets...")
        train_ds = torch.load(train_processed_path)
        val_ds = torch.load(val_processed_path)
    else:
        print("Preprocessing datasets...")
        train_ds = raw["train"].map(preprocess_fn, batched=False)
        val_ds = raw["validation"].map(preprocess_fn, batched=False)
        train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        # Save preprocessed datasets
        os.makedirs(output_dir, exist_ok=True)
        torch.save(train_ds, train_processed_path)
        torch.save(val_ds, val_processed_path)


    model_save_path = os.path.join(output_dir, f"qlora_squad_seed_{seed}")

    if not skip_training:
        # Setup Config for BitsAndBytesConfig to make the model quantizable
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

        # Prepare model for k-bit training
        model = BertForQuestionAnswering.from_pretrained(base_model, quantization_config=bnb_config, device_map="auto")
        model = prepare_model_for_kbit_training(model)

        # Model + LoRA setup
        lora_cfg = LoraConfig(
            r=8, lora_alpha=16,
            target_modules=["query","value"],
            lora_dropout=0.1,
            bias="none",
            task_type="QUESTION_ANS",
            use_rslora=False,
        )

        model = get_peft_model(model, lora_cfg)

        # Training arguments
        train_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"seed_{seed}"),
            per_device_train_batch_size=batch_size,
            num_train_epochs=num_epochs,
            learning_rate=learning_rate,
            fp16=torch.cuda.is_available(),
            logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
            logging_strategy="steps",
            logging_steps=500,
            save_strategy="epoch",
            eval_strategy="epoch",
            report_to="none",
            label_names=["start_positions", "end_positions"],
            gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
            gradient_checkpointing_kwargs={"use_reentrant": False},  # Use non-reentrant gradient checkpointing
            max_grad_norm=None,  # Gradient clipping to avoid exploding gradients
        )
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=default_data_collator,
        )

        # Training
        print(f"Training with seed {seed}...")
        trainer.train()

        # Check peak memory
        peak_mem = None
        if torch.cuda.is_available():
            peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
            print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

        # Save model and tokenizer
        trainer.save_model(model_save_path)
        tokenizer.save_pretrained(model_save_path)

        # Cleanup training objects
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()


    # Evaluate
    print("Evaluating model...")
    # Load LoRA-wrapped model for inference
    eval_base_model = BertForQuestionAnswering.from_pretrained(base_model)

    # Check if the saved model exists before loading
    if not os.path.exists(model_save_path):
        raise FileNotFoundError(f"Model not found at {model_save_path}. Please train the model first.")

    infer_model = PeftModel.from_pretrained(eval_base_model, model_save_path)


    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    evaluation_details = [] # List to store evaluation details


    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predicted_answer = out.get("answer", "")
        predictions.append({"id": item["id"], "prediction_text": predicted_answer})
        references.append({"id": item["id"], "answers": item["answers"]})

        # Store evaluation details
        evaluation_details.append({
            "id": item["id"],
            "context": item["context"],
            "question": item["question"],
            "predicted_answer": predicted_answer,
            "gold_answer": item["answers"]["text"][0] if item["answers"]["text"] else "" # Assuming there's at least one gold answer
        })

    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del infer_model, eval_base_model
    gc.collect()

    # Save results to a JSON file
    results_file_path = os.path.join(output_dir, f"qlora_results_seed_{seed}.json")
    with open(results_file_path, "w") as f:
        json.dump(results, f, indent=4)

    # Save evaluation details to a JSON file
    evaluation_details_path = os.path.join(output_dir, f"qlora_evaluation_details_seed_{seed}.json")
    with open(evaluation_details_path, "w") as f:
        json.dump(evaluation_details, f, indent=4)

    return results

#### Seed 42 ✅

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=42)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 42...


Epoch,Training Loss,Validation Loss
1,2.603000,2.302571
2,2.278700,2.060738
3,2.191700,2.000760


Peak CUDA memory (GB): 1.31
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 57.19 / 67.76
{'exact_match': 57.19016083254494, 'f1': 67.76317587617797}


#### Seed 1234 ✅


In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=1234)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 1234...


Epoch,Training Loss,Validation Loss
1,2.585800,2.292666
2,2.286300,2.070626
3,2.202800,2.014961


Peak CUDA memory (GB): 1.31
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 57.22 / 67.68
{'exact_match': 57.21854304635762, 'f1': 67.68018497020275}


#### Seed 2023 ✅

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2023)
print(results)

Preprocessing datasets...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 2023...


Epoch,Training Loss,Validation Loss
1,2.609300,2.315607
2,2.288600,2.063786
3,2.188600,2.002766


Peak CUDA memory (GB): 1.31
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 57.92 / 67.74
{'exact_match': 57.918637653736994, 'f1': 67.74027429838556}


#### Seed 2024 ✅

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2024)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 2024...


Epoch,Training Loss,Validation Loss
1,2.580700,2.300120
2,2.280800,2.065314
3,2.191500,2.003845


Peak CUDA memory (GB): 1.31
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 57.42 / 67.47
{'exact_match': 57.41721854304636, 'f1': 67.46834427990497}


#### Seed 2025 ✅

In [ ]:
# Use the function to run experiments with different seeds
results = run_qlora_experiment(seed=2025)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Peak CUDA memory (GB): 1.31


/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1bdf3ad7-df8d-4291-82f3-1c4bf1aeed0c)') - silently ignoring the lookup for the file config.json in bert-base-uncased.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in bert-base-uncased - will assume that the vocabulary was not modified.
  warnings.warn(


Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Using the latest cached version of the dataset since squad couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /root/.cache/huggingface/datasets/squad/plain_text/0.0.0/7b6d24c440a36b6815f21b70d25016731768db1f (last modified on Tue Jun 24 02:57:06 2025).
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 54.25 / 64.49
{'exact_match': 54.24787133396405, 'f1': 64.49439064396726}


### Training BERT with RoRA for Question Answering

#### Main Experiment Function

In [ ]:
# === qa_rora_squad.py ===
import os
import evaluate
import numpy as np
import torch
import gc
import json # Import json

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
    pipeline,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
)


def run_rora_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    logging_dir: str = "./logs",
    num_epochs: int = 10,
    batch_size: int = 32,
    learning_rate: float = 5e-5,
    device: int = 0,  # GPU device index, -1 for CPU
    skip_training: bool = False, # Added parameter to skip training
):
    """
    Run a full train + evaluate pipeline with LoRA on a QA task for a given seed.

    Parameters:
    - seed: Random seed for reproducibility.
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Root directory for saving model & logs.
    - logging_dir: Directory for training logs.
    - num_epochs: Number of training epochs.
    - batch_size: Training batch size.
    - learning_rate: Initial learning rate.
    - device: GPU device index or -1 for CPU.
    - skip_training: If True, skips training and proceeds directly to evaluation.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Reset GPU memory stats if using GPU
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()  # Reset peak memory stats if using GPU
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Preprocessing function
    def preprocess_fn(ex):
        tok = tokenizer(
            ex["question"], ex["context"],
            truncation="only_second",
            max_length=384,
            stride=128,
            return_overflowing_tokens=False,
            return_offsets_mapping=True,
            padding="max_length",
        )
        offsets = tok.pop("offset_mapping")
        start_char = ex["answers"]["answer_start"][0]
        end_char = start_char + len(ex["answers"]["text"][0])
        start_idx = end_idx = 0
        for i, (s, e) in enumerate(offsets):
            if s <= start_char < e:
                start_idx = i
            if s < end_char <= e:
                end_idx = i
                break
        tok["start_positions"] = start_idx
        tok["end_positions"] = end_idx
        return tok

    # Load and preprocess dataset
    raw = load_dataset(base_dataset)

    # Check if preprocessed files exist before mapping
    train_processed_path = os.path.join(output_dir, f"rora_train_ds_seed_{seed}.pt")
    val_processed_path = os.path.join(output_dir, f"rora_val_ds_seed_{seed}.pt")

    if os.path.exists(train_processed_path) and os.path.exists(val_processed_path):
        print("Loading preprocessed datasets...")
        train_ds = torch.load(train_processed_path, weights_only=False)
        val_ds = torch.load(val_processed_path, weights_only=False)
    else:
        print("Preprocessing datasets...")
        train_ds = raw["train"].map(preprocess_fn, batched=False)
        val_ds = raw["validation"].map(preprocess_fn, batched=False)
        train_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        val_ds.set_format(type="torch", columns=["input_ids","attention_mask","start_positions","end_positions"])
        # Save preprocessed datasets
        os.makedirs(output_dir, exist_ok=True)
        torch.save(train_ds, train_processed_path)
        torch.save(val_ds, val_processed_path)


    # Model + LoRA setup
    model = BertForQuestionAnswering.from_pretrained(base_model)
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["query","value"],
        lora_dropout=0.1,
        bias="none",
        task_type="QUESTION_ANS",
        use_rslora=True,
    )

    model_save_path = os.path.join(output_dir, f"rora_squad_seed_{seed}")

    if not skip_training:
        model = get_peft_model(model, lora_cfg)

        # Training arguments
        train_args = TrainingArguments(
            output_dir=os.path.join(output_dir, f"seed_{seed}"),
            per_device_train_batch_size=batch_size,
            num_train_epochs=num_epochs,
            learning_rate=learning_rate,
            fp16=torch.cuda.is_available(),
            logging_dir=os.path.join(logging_dir, f"seed_{seed}"),
            logging_strategy="steps",
            logging_steps=500,
            save_strategy="epoch",
            eval_strategy="epoch",
            report_to="none",
            label_names=["start_positions", "end_positions"],
        )
        trainer = Trainer(
            model=model,
            args=train_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=default_data_collator,
        )

        # Training
        print(f"Training with seed {seed}...")
        trainer.train()

        # Check peak memory
        peak_mem = None
        if torch.cuda.is_available():
            peak_mem = torch.cuda.max_memory_allocated() / (1024**3)  # in GB
            print(f"Peak CUDA memory (GB): {peak_mem:.2f}")

        # Save model and tokenizer
        trainer.save_model(model_save_path)
        tokenizer.save_pretrained(model_save_path)

        # Cleanup training objects
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    # Evaluate
    print("Evaluating model...")
    # Load RoRA-wrapped model for inference
    base = BertForQuestionAnswering.from_pretrained(base_model)

    # Check if the saved model exists before loading
    if not os.path.exists(model_save_path):
        raise FileNotFoundError(f"Model not found at {model_save_path}. Please train the model first.")

    infer_model = PeftModel.from_pretrained(base, model_save_path)

    qa_pipe = pipeline("question-answering", model=infer_model, tokenizer=tokenizer, device=device)
    dataset = load_dataset(base_dataset, split="validation")
    predictions, references = [], []
    evaluation_details = [] # List to store evaluation details

    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predicted_answer = out.get("answer", "")
        predictions.append({"id": item["id"], "prediction_text": predicted_answer})
        references.append({"id": item["id"], "answers": item["answers"]})

        # Store evaluation details
        evaluation_details.append({
            "id": item["id"],
            "context": item["context"],
            "question": item["question"],
            "predicted_answer": predicted_answer,
            "gold_answer": item["answers"]["text"][0] if item["answers"]["text"] else "" # Assuming there's at least one gold answer
        })

    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)
    print(f"Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del infer_model, base
    gc.collect()

    # Save results to a JSON file
    results_file_path = os.path.join(output_dir, f"rora_results_seed_{seed}.json")
    with open(results_file_path, "w") as f:
        json.dump(results, f, indent=4)

    # Save evaluation details to a JSON file
    evaluation_details_path = os.path.join(output_dir, f"rora_evaluation_details_seed_{seed}.json")
    with open(evaluation_details_path, "w") as f:
        json.dump(evaluation_details, f, indent=4)

    return results

#### Seed 42 (10 epoch) ✅

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=42)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 42...


Epoch,Training Loss,Validation Loss
1,2.447600,2.188028
2,2.065500,1.843317
3,1.859900,1.690607
4,1.736000,1.618282
5,1.677600,1.559999
6,1.599500,1.515303
7,1.570400,1.493557
8,1.549200,1.482068
9,1.548100,1.469763
10,1.528400,1.468737


Peak CUDA memory (GB): 4.36
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 71.63 / 80.25
{'exact_match': 71.62724692526017, 'f1': 80.24539956643382}


#### Seed 1234 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=1234)
print(results)

Preprocessing datasets...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 1234...


Epoch,Training Loss,Validation Loss
1,2.365600,2.067923
2,2.048400,1.843527
3,1.948100,1.783503


Peak CUDA memory (GB): 4.32
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 63.07 / 73.23
{'exact_match': 63.074739829706715, 'f1': 73.23465663579915}


#### Seed 2023 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2023)
print(results)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Training with seed 2023...


Epoch,Training Loss,Validation Loss
1,2.448800,2.156228
2,2.114000,1.901981
3,2.008400,1.840385


Peak CUDA memory (GB): 4.39
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Results (EM / F1): 61.87 / 71.70
{'exact_match': 61.873226111636704, 'f1': 71.70422760446262}


#### Seed 2024 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2024)
print(results)

Preprocessing datasets...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 2024...


Epoch,Training Loss,Validation Loss
1,2.405800,2.114992
2,2.070300,1.870241
3,1.972800,1.820544


Peak CUDA memory (GB): 4.46
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 63.25 / 73.29
{'exact_match': 63.24503311258278, 'f1': 73.29405850002327}


#### Seed 2025 ✅

In [ ]:
# Run the experiment with a specific seed
results = run_rora_experiment(seed=2025)
print(results)

Preprocessing datasets...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training with seed 2025...


Epoch,Training Loss,Validation Loss
1,2.355000,2.059695
2,2.039500,1.850124
3,1.948800,1.797339


Peak CUDA memory (GB): 4.32
Evaluating model...


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Results (EM / F1): 63.44 / 73.36
{'exact_match': 63.443708609271525, 'f1': 73.36365546346168}


# Google Drive saving

To save the training data and evaluation results to your Google Drive, we first need to mount your Drive. This will allow the notebook to access files in your Drive.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Now that your Google Drive is mounted, you can move the training data and evaluation results to a specified folder in your Drive. I will create a code cell to do this.

In [7]:
import os
import shutil

# Define the local directory where results are saved
local_results_dir = "./results"

# Define the target directory in your Google Drive
# You can change 'My Drive/Thesis_Results' to your desired path in Google Drive
google_drive_results_dir = "/content/drive/My Drive/Thesis_Results"

# Create the target directory in Google Drive if it doesn't exist
os.makedirs(google_drive_results_dir, exist_ok=True)

# Move the contents of the local results directory to Google Drive
for item in os.listdir(local_results_dir):
    s = os.path.join(local_results_dir, item)
    d = os.path.join(google_drive_results_dir, item)
    if os.path.isdir(s):
        shutil.move(s, d)
    else:
        shutil.move(s, d)

print(f"Moved contents of {local_results_dir} to {google_drive_results_dir}")

Moved contents of ./results to /content/drive/My Drive/Thesis_Results


# Pengujian base model pada tugas question answering squad

In [ ]:
# Menguji Model Dasar BERT (bert-base-uncased)
# perlu dicatat, model BERT tanpa fine-tuning tidak bisa memenuhi tugas Question Answering
# Itu juga alasan kenapa pilih BERT yang pre-trained. Jadi hasil apapun dari pipeline ini,
# tidak mempengaruhi perbandingan teknik fine tuning

import os
import evaluate
import numpy as np
import torch
import gc
import json

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForQuestionAnswering,
    pipeline,
    set_seed,
)

def run_base_model_experiment(
    seed: int,
    base_model: str = "bert-base-uncased",
    base_dataset: str = "squad",
    output_dir: str = "./results",
    device: int = 0,  # GPU device index, -1 for CPU
):
    """
    Run evaluation of the base BERT model on a QA task for a given seed.

    Parameters:
    - seed: Random seed (used for consistency, though not critical for pure inference).
    - base_model: Hugging Face model identifier.
    - base_dataset: Hugging Face dataset identifier.
    - output_dir: Directory for saving results.
    - device: GPU device index or -1 for CPU.
    """
    # Set seed
    set_seed(seed)

    # Prepare tokenizer
    tokenizer = BertTokenizerFast.from_pretrained(base_model)

    # Set device
    if device >= 0:
        torch.cuda.set_device(device)
    else:
        device = -1

    # Load the base model
    model = BertForQuestionAnswering.from_pretrained(base_model)

    # Create a question answering pipeline
    qa_pipe = pipeline("question-answering", model=model, tokenizer=tokenizer, device=device)

    # Load the validation dataset
    dataset = load_dataset(base_dataset, split="validation")

    predictions, references = [], []
    evaluation_details = [] # List to store evaluation details

    print("Evaluating base model...")
    for item in dataset:
        out = qa_pipe(question=item["question"], context=item["context"])
        predicted_answer = out.get("answer", "")
        predictions.append({"id": item["id"], "prediction_text": predicted_answer})
        references.append({"id": item["id"], "answers": item["answers"]})

        # Store evaluation details
        evaluation_details.append({
            "id": item["id"],
            "context": item["context"],
            "question": item["question"],
            "predicted_answer": predicted_answer,
            "gold_answer": item["answers"]["text"][0] if item["answers"]["text"] else "" # Assuming there's at least one gold answer
        })


    # Compute metrics
    metric = evaluate.load(base_dataset)
    results = metric.compute(predictions=predictions, references=references)

    print(f"Base Model Results (EM / F1): {results['exact_match']:.2f} / {results['f1']:.2f}")

    # Cleanup
    torch.cuda.empty_cache()
    del model
    gc.collect()

    # Save results to a JSON file
    results_file_path = os.path.join(output_dir, f"base_model_results_seed_{seed}.json")
    os.makedirs(output_dir, exist_ok=True)
    with open(results_file_path, "w") as f:
        json.dump(results, f, indent=4)

    # Save evaluation details to a JSON file
    evaluation_details_path = os.path.join(output_dir, f"base_model_evaluation_details_seed_{seed}.json")
    with open(evaluation_details_path, "w") as f:
        json.dump(evaluation_details, f, indent=4)


    return results

# Run the base model evaluation with a specific seed
# You can choose any seed value, as it mainly affects data shuffling if applicable,
# but for pure inference on the validation set, the seed doesn't significantly impact the result.
base_model_results = run_base_model_experiment(seed=42)
print(base_model_results)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Evaluating base model...
Base Model Results (EM / F1): 0.50 / 6.46
{'exact_match': 0.5014191106906338, 'f1': 6.4608395615281555}
